In [43]:
import numpy as np
import matplotlib.pyplot as plt
import scipy 
from pfapack import pfaffian as pf
import h5py
import time
import os
import traceback

This notebook implements the **Pfaffian Quantum Monte Carlo (PQMC)** method with **Hybrid Monte Carlo (HMC)** for the **Kitaev chain** with two fermion species and a repulsive on-site interaction.

The core tasks are:

1. Compute the **weight** of a given auxiliary-field configuration, which involves Pfaffians.
2. Compute the **gradient** of this weight with respect to the auxiliary-field variables.

Both the weight and its gradient can be evaluated in multiple ways. The two main approaches are:

- **Large-matrix approach:** compute the weight from a single Pfaffian of one large matrix.
- **Caterpillar approach:** compute the weight as a product of Pfaffians of much smaller matrices.

In the following, we refer to the second method as the **caterpillar** approach.

> **Remark.** The caterpillar structure is not the same as the “sausage” expression in DQMC.  
> In DQMC, one only needs a single determinant of a long product of matrices.  
> Here, by contrast, we need many Pfaffians, together with additional matrix multiplications.

We work with the Hamiltonian

$$
H
= - \sum_{\langle i,j\rangle,\sigma}
\left(
t\, a_{i,\sigma}^\dagger a_{j,\sigma}
+ \Delta\, a_{i,\sigma}^\dagger a_{j,\sigma}^\dagger
+ \mathrm{H.c.}
\right)
- \mu \sum_{x,\sigma} n_{x,\sigma}
+ U \sum_x
\left(n_{x,\uparrow}-\frac{1}{2}\right)
\left(n_{x,\downarrow}-\frac{1}{2}\right).
$$

After gaugetransforming the odd latice sites and substituting

$$
a_{i,\sigma} = \frac{1}{2} (\eta_{i,\sigma} + i \gamma_{i,\sigma}) \quad \quad \mathrm{ and }\quad \quad a_{i,\sigma}^\dagger = \frac{1}{2} (\eta_{i,\sigma} - i \gamma_{i,\sigma}) \quad \Rightarrow \left\{\eta_i, \eta_j \right\} = \left\{ \gamma_i, \gamma_j\right\} = 2\delta_{i,j},\quad \quad \left\{ \gamma_i, \eta_j\right\} = 0
$$

we end up with

$$
H = -\frac{i}{2}\sum_{<i,j>,\sigma} (t-\Delta) \eta_{i,\sigma}\eta_{j\sigma} + (t + \Delta) \gamma_{i\sigma}\gamma_{j\sigma} - \frac{i}{2}\mu \sum_{i,\sigma}\eta_{i,\sigma}\gamma_{i,\sigma} - \frac{U}{4}\sum_{i}\eta_{i,\uparrow}\gamma_{i,\uparrow}\eta_{i,\downarrow}\gamma_{i,\downarrow} = H_0 + H_1
$$

To decouple the problem we rewrite

$$
\eta_\uparrow\gamma_\uparrow\eta_\downarrow\gamma_\downarrow = 1 + \frac{1}{2}\left( \eta_\uparrow \gamma_\uparrow + \eta_\downarrow \gamma_\downarrow \right).
$$

Note, that this will lead to a real HS transformation, and a different choice is possible. This way we decouple up and down spins. Another choice would attempt to decouple the majorana types, this works for the interaction term, however, if we have finite chemical potential then we always have a coupling between Majorana types. After trotterization we find

$$
Z = \mathrm{Tr}\left\{ \prod_{t= 1}^{N_\tau} e^{-\Delta_\tau H_0} e^{+\Delta_\tau \frac{U}{8}} \sum_{i}\left( \eta_{i,\uparrow} \gamma_{i,\uparrow} + \eta_{i,\downarrow} \gamma_{i,\downarrow}  \right) \right\}.
$$

After a HS-trafo we end up with

$$
Z = \int \mathcal{D}\left( \phi \right) \mathrm{exp}\left( - \frac{2\phi^2}{\Delta_\tau U} \right) \mathrm{Tr}\left\{ e^{-\Delta_\tau H_0} \mathrm{exp}\left( - \sum_{i} \phi_i^t \left( \eta_{i,\uparrow} \gamma_{i,\uparrow} + \eta_{i,\downarrow} \gamma_{i,\downarrow} \right) \right)\right\}
$$

To use the machinery of pfQMC we rewrite this as
$$
Z = \int \mathcal{D}\left( \phi \right) \mathrm{exp}\left( - \frac{2\phi^2}{\Delta_\tau U} \right) \mathrm{Tr}\left\{ \mathrm{exp}\left( -\frac{1}{4} \boldsymbol{\theta}^T_t \boldsymbol{A}_{2t-1} \boldsymbol{\theta}_t \right) \mathrm{exp}\left( -\frac{1}{4} \boldsymbol{\theta}^T_t \boldsymbol{A}_{2t} \boldsymbol{\theta}_t \right)\right\}
$$

with

$$
\boldsymbol{\theta}_t = \left(\eta_{1,\uparrow}^t,...,\eta_{N,\uparrow}^t,\gamma_{1,\uparrow}^t,...,\gamma_{N,\uparrow},\eta_{1,\downarrow}^t,...,\eta_{N,\downarrow}^t,\gamma_{1,\downarrow}^t,...,\gamma_{N,\downarrow}, \right)^T
$$

,

$$
A_{2t} = 2\Delta_\tau \begin{bmatrix}
0 & -A_\uparrow^t & 0 & 0 \\
A_\uparrow^t & 0 & 0 & 0 \\
0 & 0 & 0 & -A_\downarrow^t \\
0 & 0 & A_\downarrow^t \\
\end{bmatrix}, \quad \quad \quad A_\uparrow = A_\downarrow = \mathrm{diag}\left( \phi_1^t,...,\phi_N^t \right)\\
$$

and

$$
A_{2t-1} = 4 \Delta_\tau H_0
$$

Because all $A_t$ are block diagonal, the up spin and downspin parts decouple and are only connected by the auxiliary field. We can therfore write the weight as a product over the two spin sectors. With these definitions we are able to calculate the weight as well as its gradient. For further details look at my notes.

In [44]:
def get_eta_B(A):
    """Takes in a skew-symmetric matrix and returns eta and B as defined in the pfaffian paper in eq.(4)"""

    B = scipy.linalg.expm(-A)

    sinhA = scipy.linalg.sinhm(A/4)

    bdim = B.shape[0]

    M = np.zeros((2*bdim, 2*bdim), dtype = np.complex128)

    M[0:bdim, 0:bdim] = np.sqrt(2) * sinhA
    M[bdim:,0:bdim]   = np.identity(bdim)
    M[0:bdim,bdim:]   = -np.identity(bdim)
    M[bdim:,bdim:]    = np.sqrt(2) * sinhA

    eta = np.power(-1,bdim//2) * pf.pfaffian(M)

    return eta, B

def get_avg_phase(filename):

    with h5py.File(filename, "r") as f:
        N_samples = len(f["data"].keys())

        phases = np.zeros(N_samples, dtype = np.complex128)

        for i in range(N_samples):
            
            path = "data/" + str(i) + "/action"

            action = f[path][()]
            phases[i] = np.exp(1j*action.imag)

    return np.mean(phases)

def multiplication(eta1, B1, eta2, B2):
    """Implementation of the contraction law presented in the pfaffian paper in eq.(5)"""

    bdim = B1.shape[0]

    M = np.zeros((2*bdim, 2*bdim), dtype = np.complex128)

    G1 = (np.identity(bdim) - B1) @ scipy.linalg.inv(np.identity(bdim) + B1)
    G2 = (np.identity(bdim) - B2) @ scipy.linalg.inv(np.identity(bdim) + B2)

    M[0:bdim, 0:bdim] = G1
    M[bdim:,0:bdim]   = np.identity(bdim)
    M[0:bdim,bdim:]   = -np.identity(bdim)
    M[bdim:,bdim:]    = G2

    eta12 = np.power(-1,bdim//2) * eta1 * eta2 * pf.pfaffian(M)

    B12   = B1@B2

    return eta12, B12

def generate_random_Alist(N, Ntau, scaling = 0.1, seed = 155351):
    """generates some random skew-symmetric matrices for testing (not relevant to the algorithm)"""
    Alist = []
    rng = np.random.default_rng(seed = seed)

    for i in range(Ntau):

        A = rng.normal(loc = 0, scale = scaling, size = (2*N, 2*N)) + 1j * rng.normal(loc = 0, scale = scaling, size = (2*N, 2*N))
        A = 0.5*(A - A.T)
        Alist.append(A)

    return Alist

def get_correlator_and_action(h5_name: str, dex: int):

    with h5py.File(h5_name, "r") as f:
        N_cfg = len(f["data"].keys())
        if dex >= N_cfg:
            print("Index too large")
            return -1
        
        path_corr   = "data/" + str(dex) + "/correlators"
        path_action = "data/" + str(dex) + "/action"

        correlator = f[path_corr][:]
        action     = f[path_action][()]
    return correlator, action

def read_correlator_file(filename, model):
    """Careful, this returns the weighted correlators, so the correlators multiplied by the phases"""
    with h5py.File(filename, 'r') as f:

        N_samples = len(f['/data'])

    correlator_samples = np.zeros((2*model.L, 2*model.L, model.Ntau+1, N_samples), dtype = np.complex128)
    phases             = np.empty(N_samples, dtype = np.complex128)
    action = 0

    for i in range(N_samples):

        correlator_samples[:,:,:,i], action = get_correlator_and_action(filename, i)
        phases[i]                           = np.exp(-1j*action.imag)
        correlator_samples[:,:,:,i]         = correlator_samples[:,:,:,i] * phases[i]

    return correlator_samples, phases

def bootstrap(samples, binsize, N_boot, rng, correlated_r_idx = None):

    dtype = samples.dtype

    N_bins = samples.shape[-1]//binsize

    if N_bins < 2:
        raise ValueError("Need at least two bins for a bootstrap error estimate.")


    sample_size = N_bins*binsize

    samples = samples[...,:sample_size]

    samples = samples.reshape(*samples.shape[:-1], N_bins, binsize)

    sample_bin_means = samples.mean(axis = -1)

    empiric_means = np.empty((*sample_bin_means.shape[:-1], N_boot), dtype = dtype)

    if correlated_r_idx is None:
        r_idx = rng.integers(low = 0, high = N_bins, size = N_bins*N_boot).reshape(N_bins,N_boot)
    else:
        r_idx = correlated_r_idx

    for i in range(N_boot):

        empiric_means[...,i] = np.mean(sample_bin_means[...,r_idx[:,i]], axis = -1)

    return sample_bin_means.mean(axis = -1), np.std(empiric_means.real, axis = -1, ddof = 1) + 1j* np.std(empiric_means.imag, axis = -1, ddof = 1), r_idx

def bootstrap_reweighting(samples, phases, binsize, N_boot, rng, correlated_r_idx = None):

    dtype = samples.dtype

    N_bins = samples.shape[-1]//binsize

    if N_bins < 2:
        raise ValueError("Need at least two bins for a bootstrap error estimate.")


    sample_size = N_bins*binsize

    samples = samples[...,:sample_size]
    phases  = phases[...,:sample_size]

    samples = samples.reshape(*samples.shape[:-1], N_bins, binsize)
    phases = phases.reshape(*phases.shape[:-1], N_bins, binsize)

    sample_bin_means = samples.mean(axis = -1)
    phases_bin_means = phases.mean(axis = -1)

    empiric_means = np.empty((*sample_bin_means.shape[:-1], N_boot), dtype = dtype)
    empiric_means_phase = np.empty((*phases_bin_means.shape[:-1], N_boot), dtype = dtype)

    if correlated_r_idx is None:
        r_idx = rng.integers(low = 0, high = N_bins, size = N_bins*N_boot).reshape(N_bins,N_boot)
    else:
        r_idx = correlated_r_idx

    for i in range(N_boot):

        empiric_means[...,i] = np.mean(sample_bin_means[...,r_idx[:,i]], axis = -1)
        empiric_means_phase[...,i] = np.mean(phases_bin_means[...,r_idx[:,i]], axis = -1)

    reweighted_mean = (sample_bin_means).mean(axis = -1)/phases_bin_means.mean()
    reweighted_std  = np.std((empiric_means/empiric_means_phase).real, axis = -1, ddof = 1) + 1j* np.std((empiric_means/empiric_means_phase).imag, axis = -1, ddof = 1)
    phase_mean      = phases.mean()
    phase_std       = phases.real.std() + 1j*phases.imag.std()

    return reweighted_mean, reweighted_std, phase_mean, phase_std

def get_dirac_correlators(correlators):

    L = correlators.shape[0]//2

    dirac_correlators = 0.25*(correlators[:L,:L] - 1j * correlators[:L,L:] +1j * correlators[L:,:L] + correlators[L:,L:])

    return dirac_correlators

class DataSaver():
    def __init__(self, filename: str, metadata: dict):
        self.path = filename
        self.file = h5py.File(filename, "w")
        self.metadata = self.file.create_group("metadata")
        for key, val in metadata.items():
            self.metadata.create_dataset(key, data = val)
        self.data = self.file.create_group("data")
        self.counter  = 0

    def save_cfg(self, cfg: np.ndarray, action: np.complex128):
        cfg_group = self.data.create_group(f"{self.counter}")
        cfg_group.create_dataset("cfg", data = cfg)
        cfg_group.create_dataset("action", data = action)
        self.counter += 1

    def save_correlators(self, correlators: np.ndarray, action = np.float64):
        cfg_group = self.data.create_group(f"{self.counter}")
        cfg_group.create_dataset("correlators", data = correlators)
        cfg_group.create_dataset("action", data = action)
        self.counter += 1

    def close(self):
        self.file.close()

class HMC():
    """Implementation of the Hybrid Monte Carlo algorithm wihtout any reweighting
        It currently uses the caterpillar form for both the weight as well as the gradient"""
    
    def __init__(self, t_MD, N_MD, model, saver, seed = 1337):
        self.N_MD = N_MD
        self.t_MD  = t_MD
        self.dt, self.dt_half = t_MD/self.N_MD, 0.5*t_MD/self.N_MD
        self.model = model
        self.cfg_shape = model.shape
        self.saver = saver
        self.steps = self.acc = 0
        self.rng   = np.random.default_rng(seed)


    def step(self):

        self.mom = self.rng.normal(loc = 0.0, scale = 1.0, size = self.cfg_shape).astype(np.float64)
        new_cfg  = np.copy(self.model.cfg)
        old_H    = self.model.action.real + 0.5 * np.sum(np.power(self.mom, 2))
        self.mom -= self.dt_half*self.model.get_action_gradient_DQMC_with_measurement_fast(new_cfg)

        for _ in range(self.N_MD-1):

            new_cfg  +=self.dt*self.mom
            self.mom -= self.dt*self.model.get_action_gradient_DQMC_with_measurement_fast(new_cfg)
        
        new_cfg   += self.dt*self.mom
        self.mom  -= self.dt_half*self.model.get_action_gradient_DQMC_with_measurement_fast(new_cfg, with_measurement = True)
        new_action = self.model.get_action_caterpillar(new_cfg)

        new_H = new_action.real + 0.5 * np.sum(np.power(self.mom, 2))
        dH    = new_H - old_H

        p = np.exp(-dH)
        r = self.rng.random()

        if r < p :

            self.acc += 1
            self.model.cfg = new_cfg
            self.model.action = new_action
            self.model.correlators[:,:,:] = self.model.proposed_correlators[:,:,:]

        self.steps += 1


    def save(self):
        self.saver.save_correlators(self.model.correlators, self.model.action)

    def output_acc(self):
        return self.acc/self.steps
        
    def reset_acc(self):
        self.acc = self.steps = 0

    def output_model(self):
        return self.model

    def close_file(self):
        self.saver.close()

class Kitaev_action():
    def __init__(self, L, U, Delta, mu, dtau, Ntau, seed = 153525):
        #set paramters and initialize configuration     
        self.U = U
        self.mu = mu
        self.dtau = dtau
        self.Ntau = Ntau
        self.Delta = Delta
        self.L = L                  #In the notes this is referred to as N
        self.Dplus =  Delta + 1
        self.Dminus = Delta - 1

        self.rng = np.random.default_rng(seed)
        self.shape = (L, Ntau)
        self.cfg = self.rng.normal(loc=0.0, scale = 1/np.sqrt(np.prod(self.shape)), size = self.shape)
        self.eye = np.eye(2*self.L)

        # build the non interacting Hamiltonian and from it the odd A-matrices
        self.A_odd = np.zeros((2*L, 2*L), dtype = np.complex128)
        # self.A_odd[:L,:L] = np.roll(np.eye(L)*self.Dminus, 1, axis = 1) - np.roll(np.eye(L)*self.Dminus, -1, axis = 1)
        # self.A_odd[L:,L:] = np.roll(np.eye(L)*self.Dplus, 1, axis = 1)  - np.roll(np.eye(L)*self.Dplus, -1, axis = 1)
        # self.A_odd[:L,L:] = np.eye(L)*mu
        # self.A_odd[L:,:L] = -np.eye(L)*mu
        # self.A_odd[L-1, 0] = 0
        # self.A_odd[0, L-1] = 0
        # self.A_odd[2*L - 1, L] = 0
        # self.A_odd[L, 2*L-1]   = 0

        self.A_odd[:L,L:]  = -np.eye(L)*mu
        self.A_odd[:L,L:] += np.roll(np.eye(L)*self.Dminus, 1, axis = 1)
        self.A_odd[:L,L:] += -np.roll(np.eye(L)*self.Dplus, -1, axis = 1)
        self.A_odd[L-1,L] = 0
        self.A_odd[0,2*L-1] = 0

        self.A_odd[L:, :L] = -self.A_odd[:L,L:].T

        self.A_odd = 1j*dtau*self.A_odd

        #self.A_odd = -self.A_odd * self.dtau * 1j

        # calculate quantities connected to the odd A_ts
        self.eta_odd, self.B_odd = get_eta_B(self.A_odd)
        self.G_odd = scipy.linalg.tanhm(self.A_odd/2)
        self.expminusA = scipy.linalg.expm(-self.A_odd)
        self.expplusA  = scipy.linalg.expm(self.A_odd)
        self.eta_odd_prod = np.power(self.eta_odd, self.Ntau)

        self.temp = np.zeros((4*self.L, 4*self.L), dtype = np.complex128)
        self.temp[2*self.L:, :2*self.L] = np.eye(2*self.L)
        self.temp[:2*self.L, 2*self.L:] = -np.eye(2*self.L)

        #build the offdiagonal parts of the large matrix used in the pfaffian (not for the catterpiller)
        self.g = np.zeros((self.Ntau*self.L*4, self.Ntau*self.L*4))

        block_size = 2 * L
        total_size = block_size * Ntau *2

        self.g = np.zeros((total_size, total_size), dtype=np.complex128)
        I      = np.eye(block_size, dtype=np.complex128)

        self.correlators          = np.zeros((self.L*2, self.L*2, self.Ntau+1), dtype = np.complex128)
        self.proposed_correlators = np.zeros((self.L*2, self.L*2, self.Ntau+1), dtype = np.complex128)

        for i in range(2*Ntau):
            for j in range(i + 1, 2*Ntau):
                # Checkerboard sign on upper triangle:
                # starts with minus for (i,j) = (0,1)
                sign = -1 if (j - i) % 2 == 1 else 1

                row_i = slice(i * block_size, (i + 1) * block_size)
                row_j = slice(j * block_size, (j + 1) * block_size)

                self.g[row_i, row_j] = sign * I
                self.g[row_j, row_i] = -sign * I  # antisymmetry

        self.action = self.get_action_caterpillar(self.cfg)

    def get_all_G_even(self, cfg):
        """Uses the analytic solution to calculate the Greensfunction more efficiently from the configuration"""

        all_G_even = np.zeros((2*self.L, 2*self.L, self.Ntau), dtype =  np.complex128)
        temp       = (np.eye(self.cfg.shape[0], dtype=np.complex128) * np.tan(cfg.T[:, None, :])).transpose(1, 2, 0)

        all_G_even[:self.L, self.L:,:] = +temp
        all_G_even[self.L:, :self.L,:] = -temp
        
        return all_G_even
    
    def get_g(self, cfg):
        """Fill in the Block diagoanl of the large Matrix in the pfaffian (not for the catterpiller)"""
        block_size = 2 * self.L
        i = np.arange(self.L)

        t = np.tan(cfg.T)   # (Ntau, L)

        for j in range(self.Ntau):
            odd_base = 2 * j * block_size
            self.g[odd_base:odd_base + block_size, odd_base:odd_base + block_size] = self.G_odd

            even_base = (2 * j + 1) * block_size
            self.g[even_base + i,         even_base + self.L + i] = +t[j]
            self.g[even_base + self.L + i, even_base + i]         = -t[j]

    def get_action_caterpillar(self, cfg):
        """calculating action by use of the catterpiller form"""
        two_to_the_L = 2**self.L

        L = self.L
        Ntau = self.Ntau
        n = 2 * L
        dtau = self.dtau
        U = self.U

        sign_eta = np.power(-1, self.L)

        action_B  = np.sum(2*cfg*cfg)/(self.dtau*self.U) # auxiliarry action

        # Iteratively calculate the pfaffian product in the catterpiller

        current_G = self.G_odd.copy()
        current_B = self.expminusA.copy()
        current_eta = self.eta_odd

        allBs = np.zeros((n, n), dtype=np.complex128)
        allGs = np.zeros((n, n), dtype=np.complex128) 

        x = cfg
        tan_x = np.tan(x)
        cos_x = np.cos(x)
        cos_2x = np.cos(2.0*x)
        sin_2x = np.sin(2.0*x)

        alletas = np.prod(cos_x, axis = 0)
        
        idx = np.arange(L)
        idxL = idx + L

        for i in range(Ntau):
            c = cos_2x[:, i]
            s = sin_2x[:, i]
            t = tan_x[:, i]

            #analytically calculate the single time Bs
            allBs.fill(0.0)
            allBs[idx,  idx]  = c
            allBs[idxL, idxL] = c
            allBs[idx,  idxL] = -s
            allBs[idxL, idx]  = +s

            allGs.fill(0.0)
            allGs[idx,  idxL] = + t
            allGs[idxL, idx]  = - t
            
            #This block implements the combination of 2 FGOs (as in pfaffian paper) for a new even factor
            self.temp[:n, :n] = current_G
            self.temp[n:, n:] = allGs[:,:]

            #DANGEROUS THIS WAS ADDED BY HAND BE CAREFULL!!!!!!!!!!!!!!!!!!!!!!!!!!!
            self.temp = 0.5*(self.temp - self.temp.T)

            check = np.abs((self.temp + self.temp.T)).max()

            if check > 1e-14:
                print(check)

            current_eta                     *= sign_eta * alletas[i] * pf.pfaffian(self.temp)
            current_B                       = current_B @ allBs
            current_G                       = scipy.linalg.solve(self.eye + current_B, self.eye - current_B, assume_a='gen')

            #In the last iteration we have to skip the add part because we started with the odd part before we enter the for loop
        
            if i != (self.Ntau - 1):
                #this block is for the odd A factors
                self.temp[:n, :n] = current_G
                self.temp[n:, n:] = self.G_odd


                 #DANGEROUS THIS WAS ADDED BY HAND BE CAREFULL!!!!!!!!!!!!!!!!!!!!!!!!!!!
                self.temp = 0.5*(self.temp - self.temp.T)

                check = np.abs((self.temp + self.temp.T)).max()

                if check > 1e-14:
                    print(check, "2")


                current_eta                     *= sign_eta * self.eta_odd * pf.pfaffian(self.temp)
                current_B                       = current_B @ self.expminusA
                current_G                       = scipy.linalg.solve(self.eye + current_B, self.eye - current_B, assume_a='gen')

        #This is the actuall target weight but for HMC we need to return -log of this, because it expects the weight exponential form (with -)
        #This looks odd because in the case of the Kitaev chain and the way I HS-transformed A^up and A^down are equal, therefore the log must appear twice.
        # Perhaps a different HS-Trafo would be better?

        weight = two_to_the_L * current_eta
        action = action_B - 2.0*np.log(weight)

        return action

    def get_action_gradient_DQMC(self, cfg):

        L = self.L
        Ntau = self.Ntau
        n = 2 * L
        dtau = self.dtau
        U = self.U

        gradient = np.zeros(cfg.shape, dtype=np.complex128)

        gradient_SB = 4*cfg/(dtau * U)
        
        cos_2x = np.cos(2*cfg)
        sin_2x = np.sin(2*cfg)

        Beven = np.zeros((n, n, Ntau), dtype=np.complex128)
        Bodd  = self.expminusA
  
        idx = np.arange(L)
        idxL = idx + L

        Kphis = np.zeros((n,n,Ntau), dtype = np.complex128)
        M     = np.eye(n, dtype = np.complex128)

        for i in range(Ntau):

            c = cos_2x[:, i]
            s = sin_2x[:, i]

            Beven[idx,  idx, i]  = c
            Beven[idxL, idxL, i] = c
            Beven[idx,  idxL, i] = -s
            Beven[idxL, idx, i]  = +s

            Kphis[:,:,i] = Bodd @ Beven[:,:,i]

            M = M @ Kphis[:,:,i]

        M = np.eye(n, dtype=np.complex128) + M


        phi2t = np.zeros((n,n,Ntau), dtype = np.complex128)
        phifromt = np.zeros((n,n,Ntau), dtype = np.complex128)

        phi2t[:,:,0] = Kphis[:,:,0]
        phifromt[:,:,-1] = np.eye(n, dtype = np.complex128)

        for i in range(Ntau - 1):

            phi2t[:,:,i+1] = phi2t[:,:,i] @ Kphis[:,:,i+1]
            phifromt[:,:,Ntau - 2 - i] = Kphis[:,:,Ntau - 1 - i] @ phifromt[:,:,Ntau - 1 - i]

        M_inv = scipy.linalg.inv(M)
        
        for i in range(Ntau):
            P              = (phifromt[:,:,i] @ M_inv @ phi2t[:,:,i])
            gradient[:,i]  = P[idxL, idx] - P[idx, idxL]

        gradient[:,:] = 2*gradient.real + gradient_SB

        return gradient
    
    def get_action_gradient_DQMC_with_measurement(self, cfg):

        L = self.L
        Ntau = self.Ntau
        n = 2 * L
        dtau = self.dtau
        U = self.U

        gradient = np.zeros(cfg.shape, dtype=np.complex128)

        gradient_SB = 4*cfg/(dtau * U)
        
        cos_2x = np.cos(2*cfg)
        sin_2x = np.sin(2*cfg)

        Beven = np.zeros((n, n, Ntau), dtype=np.complex128)
        Bodd  = self.expminusA
  
        idx = np.arange(L)
        idxL = idx + L

        Kphis = np.zeros((n,n,Ntau), dtype = np.complex128)
        M     = np.eye(n, dtype = np.complex128)

        for i in range(Ntau):

            c = cos_2x[:, i]
            s = sin_2x[:, i]

            Beven[idx,  idx, i]  = c
            Beven[idxL, idxL, i] = c
            Beven[idx,  idxL, i] = -s
            Beven[idxL, idx, i]  = +s

            Kphis[:,:,i] = Bodd @ Beven[:,:,i]

            M = M @ Kphis[:,:,i]

        M = np.eye(n, dtype=np.complex128) + M


        phi2t = np.zeros((n,n,Ntau), dtype = np.complex128)
        phifromt = np.zeros((n,n,Ntau), dtype = np.complex128)

        phi2t[:,:,0] = Kphis[:,:,0]
        phifromt[:,:,-1] = np.eye(n, dtype = np.complex128)

        for i in range(Ntau - 1):

            phi2t[:,:,i+1] = phi2t[:,:,i] @ Kphis[:,:,i+1]
            phifromt[:,:,Ntau - 2 - i] = Kphis[:,:,Ntau - 1 - i] @ phifromt[:,:,Ntau - 1 - i]

        M_inv = scipy.linalg.inv(M)

        self.correlators[:,:,0] = M_inv
        
        for i in range(Ntau):
            #P              = (phifromt[:,:,i] @ M_inv @ phi2t[:,:,i])

            P              = M_inv @ phi2t[:,:,i]

            self.correlators[:,:,i+1] = 2*P

            P              = phifromt[:,:,i] @ P

            gradient[:,i] += P[idxL, idx] - P[idx, idxL]

        gradient[:,:] = 2*gradient.real + gradient_SB

        return gradient
    
    def get_action_gradient_DQMC_with_measurement_fast(self, cfg, with_measurement = False):

        L = self.L
        Ntau = self.Ntau
        n = 2 * L
        dtau = self.dtau
        U = self.U

        gradient = np.zeros(cfg.shape)

        gradient_SB = 4*cfg/(dtau * U)
        
        cos_2x = np.cos(2*cfg)
        sin_2x = np.sin(2*cfg)

        Beven = np.zeros((n, n, Ntau), dtype=np.complex128)
        Bodd  = self.expminusA
  
        idx = np.arange(L)
        idxL = idx + L

        Kphis = np.zeros((n,n,Ntau), dtype = np.complex128)
        phi2t = np.zeros((n,n,Ntau), dtype = np.complex128)
        M     = np.eye(n, dtype = np.complex128)

        for i in range(Ntau):

            c = cos_2x[:, i]
            s = sin_2x[:, i]

            Beven[idx,  idx, i]  = c
            Beven[idxL, idxL, i] = c
            Beven[idx,  idxL, i] = -s
            Beven[idxL, idx, i]  = +s

            Kphis[:,:,i] = Bodd @ Beven[:,:,i]

            M = M @ Kphis[:,:,i]
            phi2t[:,:,i] = M

        M = np.eye(n, dtype=np.complex128) + M


        phifromt = np.zeros((n,n,Ntau), dtype = np.complex128)
        phifromt[:,:,-1] = np.eye(n, dtype = np.complex128)

        for i in range(Ntau - 1):

            phifromt[:,:,Ntau - 2 - i] = Kphis[:,:,Ntau - 1 - i] @ phifromt[:,:,Ntau - 1 - i]

        M_inv = scipy.linalg.inv(M)

        if with_measurement:

            self.proposed_correlators[:,:,0] = M_inv
            
            for i in range(Ntau):
                #P              = (phifromt[:,:,i] @ M_inv @ phi2t[:,:,i])

                P              = M_inv @ phi2t[:,:,i]

                self.proposed_correlators[:,:,i+1] = P

            for i in range(Ntau):

                P              = phifromt[:,:,i] @ self.proposed_correlators[:,:,i+1]

                gradient[:,i]  = (P[idxL, idx] - P[idx, idxL]).real

            self.proposed_correlators *= 2

        else:

            for i in range(Ntau):

                P              = (phifromt[:,:,i] @ M_inv @ phi2t[:,:,i])
                gradient[:,i]  = (P[idxL, idx] - P[idx, idxL]).real


        gradient[:,:] = 2*gradient + gradient_SB

        return gradient
    
    def get_action_gradient_DQMC_fast(self, cfg):

        L = self.L
        Ntau = self.Ntau
        n = 2 * L
        dtau = self.dtau
        U = self.U

        idx = np.arange(L)
        idxL = idx + L

        gradient_SB = 4.0 * cfg / (dtau * U)

        cos_2x = np.cos(2.0 * cfg)
        sin_2x = np.sin(2.0 * cfg)

        Bodd = self.expminusA

        # Kphis[:, :, t] = Bodd @ Beven[t]
        # but constructed by cheap column mixing instead of dense matmul.
        Kphis = np.empty((n, n, Ntau), dtype=np.complex128)

        Bodd_left = Bodd[:, idx]
        Bodd_right = Bodd[:, idxL]

        for t in range(Ntau):
            c = cos_2x[:, t]
            s = sin_2x[:, t]

            K = Kphis[:, :, t]

            # First L columns:
            # Beven[:, x] = c_x e_x + s_x e_{L+x}
            K[:, idx] = Bodd_left * c[None, :] + Bodd_right * s[None, :]

            # Last L columns:
            # Beven[:, L+x] = -s_x e_x + c_x e_{L+x}
            K[:, idxL] = -Bodd_left * s[None, :] + Bodd_right * c[None, :]

        # Prefix products A(k) = K_0 ... K_k
        phi2t = np.empty((n, n, Ntau), dtype=np.complex128)
        phi2t[:, :, 0] = Kphis[:, :, 0]

        for t in range(1, Ntau):
            phi2t[:, :, t] = phi2t[:, :, t - 1] @ Kphis[:, :, t]

        # M = I + K_0 ... K_{Ntau-1}
        M = np.eye(n, dtype=np.complex128) + phi2t[:, :, -1]

        # Suffix products B(k+1) = K_{k+1} ... K_{Ntau-1}
        phifromt = np.empty((n, n, Ntau), dtype=np.complex128)
        phifromt[:, :, -1] = np.eye(n, dtype=np.complex128)

        for t in range(Ntau - 2, -1, -1):
            phifromt[:, :, t] = Kphis[:, :, t + 1] @ phifromt[:, :, t + 1]

        # Factor M once. Avoid explicit inverse.
        lu, piv = scipy.linalg.lu_factor(M, check_finite=False)

        gradient = np.empty((L, Ntau), dtype=np.float64)

        for t in range(Ntau):
            # X = M^{-1} A(t)
            X = scipy.linalg.lu_solve(
                (lu, piv),
                phi2t[:, :, t],
                check_finite=False,
            )

            S = phifromt[:, :, t]

            # We need:
            # diag([S X]_{2,1}) - diag([S X]_{1,2})
            #
            # (S X)[L+x, x]   = sum_a S[L+x, a] X[a, x]
            # (S X)[x, L+x]   = sum_a S[x, a]   X[a, L+x]

            block_21_diag = np.einsum(
                "xa,ax->x",
                S[idxL, :],
                X[:, idx],
                optimize=True,
            )

            block_12_diag = np.einsum(
                "xa,ax->x",
                S[idx, :],
                X[:, idxL],
                optimize=True,
            )

            gradient[:, t] = (block_21_diag - block_12_diag).real

        # Factor 2 because both spin sectors are identical in the setup.
        gradient = 2.0 * gradient + gradient_SB

        return gradient
  
    def get_action_gradient(self, cfg):
        """This calculates the gradient of the weight without the caterpillar form, it is therefore more computationally expensive.
        The explicit formula can be found in my notes. This however returns values that coincide with numerical differentiation and I therefore assume it is correct"""

        dtau = self.dtau
        L = self.L
        Ntau = self.Ntau
        block_size = 2 * L

        x = cfg
        tan_x = np.tan(x)
        sec2_x = 1.0 + tan_x * tan_x

        gradient_p1 = -tan_x

        self.get_g(cfg)

        t_idx = np.arange(Ntau)[:, None]
        x_idx = np.arange(L)[None, :]

        rows = ((2 * t_idx + 1) * block_size + x_idx).ravel()
        cols = (rows + L).ravel()

        # Solve only for needed columns of g^{-1}
        E = np.eye(self.g.shape[0], dtype=self.g.dtype)[:, cols]
        X = scipy.linalg.solve(self.g, E, assume_a='gen')

        g_inv_diag = X[rows, np.arange(rows.size)].reshape(Ntau, L).T
        #sign because I extract upper right quadrant
        gradient_p2 = -g_inv_diag * sec2_x

        gradient_S_B = 4.0 * cfg / (dtau * self.U)

        #HMC again expects the gradient of the negative action, not the weight, the factor 2.0 again comes from the block diagonal nature of system in our particular HS-trafo

        gradient = -2.0 * (gradient_p1 + gradient_p2).real + gradient_S_B
        return gradient
    
    def get_action_gradient_caterpillar(self, cfg):
        """This calculates the the gradient directly from the caterpillar. It is therefore more efficient.
        For an explicit form look at the notes. The values match with the previous gradient."""

        L = self.L
        Ntau = self.Ntau
        n = 2 * L
        dtau = self.dtau

        x = cfg
        tan_x = np.tan(x)
        cos_2x = np.cos(2*x)
        sin_2x = np.sin(2*x)
        tan_sq_p1 = 1 + tan_x*tan_x

        gradient = 2*tan_x

        gradient += 4*cfg/(dtau * self.U)

        S = np.eye(n, dtype = np.complex128)
        Gks = self.get_all_G_even(cfg)

        # Calculate all the even B-factors and their product (called S here)
        Beven = np.zeros((n, n), dtype=np.complex128)
        idx = np.arange(L)
        idxL = idx + L

        Beven = np.zeros((n, n, Ntau), dtype=np.complex128)
  
        for i in range(Ntau):

            c = cos_2x[:, i]
            s = sin_2x[:, i]

            Beven[idx,  idx, i]  = c
            Beven[idxL, idxL, i] = c
            Beven[idx,  idxL, i] = -s
            Beven[idxL, idx, i]  = +s

            S = S @ self.expminusA
            S = S @ Beven[:,:,i]

        currentB = S
        B_i = np.eye(n, dtype=np.complex128)

        # use the recursive formula for the different imaginary times (see notes)

        for i in range(Ntau):

            currentB = self.expplusA @ currentB @ B_i
            B_i      = self.expminusA

            currentB = Beven[:,:,i].T @ currentB @ B_i
            B_i      = Beven[:,:,i]

            Denominator = self.eye + currentB
            Enumerator  = self.eye - currentB

            M = Gks[:, :, i] @ Enumerator + Denominator
            temp_grad = scipy.linalg.solve(M.T, Enumerator.T, assume_a='gen').T

            temp_grad = np.diagonal(temp_grad[:L,L:]) * tan_sq_p1[:,i]
        
            gradient[:,i] += 2.0*temp_grad.real
            
        return gradient

def generate(algorithm, N_therm, N_cfg, save_frequency, close_file = True, out_freq = 1000):

    print(f'Begin thermilization for {N_therm} configurations.')
    t1 = time.time()
    for i in range(N_therm):
        algorithm.step()
    t2 = time.time()
    algorithm.reset_acc()
    print(f"Thermilization done after {np.round(t2-t1, 2)}s!")
    print(f"Begin generation of {N_cfg} configurations.")
    for i in range(N_cfg):
        algorithm.step()
        if i % save_frequency == 0:
            algorithm.save()
        if i % out_freq == 1:
            acc = algorithm.output_acc()
            print(f'At sample {i} out of {N_cfg}, running since {np.round(time.time()-t2, 2)}s!, projected time left: {(N_cfg/i-1)*np.round(time.time()-t2, 2)}s')
            print(f'Acceptance rate at {acc}')
    print(f"Total time for sampling: {np.round(time.time()-t2, 2)}s with an acceptance rate of {np.round(algorithm.output_acc(), 3)}!")
    if close_file:
        algorithm.close_file()

    acc = algorithm.output_acc()

    return acc


In [52]:
L = 4
U = 1.0
Delta = 0.7
mu = -0.3
dtau = 0.1
Ntau = 8
t_MD = 0.6
N_MD = 8
N_therm = 1000
N_cfg = 5000
save_frequency = 1
filename = "test.h5"
taus = np.arange(0, dtau*(Ntau+1), dtau)

sim_dict = {"L" : L,
            "U" : U,
            "Delta" : Delta,
            "mu" : mu,
            "dtau" : dtau,
            "Ntau" : Ntau,
            "t_MD" : t_MD,
            "N_MD" : N_MD,
            "N_therm" : N_therm,
            "N_cfg" : N_cfg,
            "save_frequency" : save_frequency,
            "filename" : filename
            }
try:
    model = Kitaev_action(L, U, Delta, mu, dtau, Ntau)
    saver = DataSaver(filename, sim_dict)
    algorithm = HMC(t_MD, N_MD, model, saver = saver, seed = 3432)

    generate(algorithm, N_therm, N_cfg, save_frequency)

    avg_phase = get_avg_phase(filename)

    print(avg_phase)

except Exception:
    print("Failure")
    traceback.print_exc()
    saver.close()
    os.remove(filename)
finally:
    saver.close()
    pass


Begin thermilization for 1000 configurations.
Thermilization done after 2.47s!
Begin generation of 5000 configurations.
At sample 1 out of 5000, running since 0.01s!, projected time left: 49.99s
Acceptance rate at 0.5
At sample 1001 out of 5000, running since 2.52s!, projected time left: 10.067412587412587s
Acceptance rate at 0.8942115768463074
At sample 2001 out of 5000, running since 5.05s!, projected time left: 7.568690654672663s
Acceptance rate at 0.8966033966033966
At sample 3001 out of 5000, running since 7.56s!, projected time left: 5.035801399533489s
Acceptance rate at 0.9020652898067955
At sample 4001 out of 5000, running since 10.08s!, projected time left: 2.516850787303175s
Acceptance rate at 0.8953023488255872
Total time for sampling: 12.6s with an acceptance rate of 0.897!
(0.986636811999713+0.0007094161767705756j)


In [53]:
binsize = 100
N_boot = 1000
seed = 551515

rng = np.random.default_rng(seed = 151511533)
Cm, phases = read_correlator_file(filename, model)
Cd = get_dirac_correlators(Cm)

C_dirac, C_dirac_err, phase_avg, phase_std = bootstrap_reweighting(Cd, phases, binsize, N_boot, rng)
print(phase_avg, phase_std)

(0.986636811999713-0.0007094161767705756j) (0.0212745411032189+0.16153851502529934j)


In [54]:
Ced  = np.load("ed_corr.npy")
taus_ed = np.load('ed_taus.npy')

fig, ax = plt.subplots(L, L, figsize = (L*4.8, L*4.8))
fig.supylabel(r"$C_{ij}(\tau)$")
fig.supxlabel(r"$\tau$")
for x in range(L):
    for y in range(L):
        cax = ax[x, y]
        cax.grid()
        cax.errorbar(taus, C_dirac[x,y,:].real, marker = '.', capsize=3, label = "pfQMC", yerr = np.real(C_dirac_err[x,y,:]), linestyle ="none")
        cax.errorbar(taus_ed, Ced[x,y,:].real,  linestyle = "--", label = "ED")
        cax.set_title(f"(i,j)=({x},{y})")


handles, labels = ax[0, 0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.94),
    ncol=2,
)

fig.tight_layout(rect=[0, 0, 1, 0.93])

plt.savefig("KitaevChain_spatial_corr_pfQMC.png", dpi = 200, bbox_inches = "tight")
plt.close()

In [272]:
#taus = np.arange(0, dtau*(Ntau+1), dtau)

Ced  = np.load("ed_corr.npy")
taus_ed = np.load('ed_taus.npy')
Cedtrot = np.load("tempCd.npy")
taus_ed_trot = np.load("temp_taus.npy")

fig, ax = plt.subplots(L, L, figsize = (L*4.8, L*4.8))
fig.supylabel(r"$C_{ij}(\tau)$")
fig.supxlabel(r"$\tau$")
for x in range(L):
    for y in range(L):
        cax = ax[x, y]
        cax.grid()
        cax.errorbar(taus, Cd[x,y,:].real, marker = '.', capsize=3, label = "pfQMC Ntau=16", yerr = np.abs(Cd_err[x,y,:]), linestyle ="none")
        cax.errorbar(taus_ed, Ced[x,y,:].real,  linestyle = "--", label = "ED")
        cax.errorbar(taus_ed_trot, Cedtrot[x,y,:].real,  marker = ".", linestyle = "none", label = "pfQMC Ntau=8")
        cax.set_title(f"(i,j)=({x},{y})")


handles, labels = ax[0, 0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.94),
    ncol=2,
)

fig.tight_layout(rect=[0, 0, 1, 0.93])

plt.savefig("KitaevChain_spatial_corr_pfQM_trot.png", dpi = 200, bbox_inches = "tight")
plt.close()

In [227]:
taus = np.arange(0, dtau*(Ntau+1), dtau)

Ced  = np.load("ed_corr.npy")
taus_ed = np.load('ed_taus.npy')
Cedtrot = np.load("ED_trotter_C.npy")
taus_ed_trot = np.load("ED_trotter_taus.npy")

fig, ax = plt.subplots(L, L, figsize = (L*4.8, L*4.8))
fig.supylabel(r"$C_{ij}(\tau)$")
fig.supxlabel(r"$\tau$")
for x in range(L):
    for y in range(L):
        cax = ax[x, y]
        cax.grid()
        cax.errorbar(taus, Cd[x,y,:].real, marker = '.', capsize=3, label = "pfQMC", yerr = np.abs(Cd_err[x,y,:]), linestyle ="none")
        cax.errorbar(taus_ed, Ced[x,y,:].real,  linestyle = "--", label = "ED")
        cax.errorbar(taus_ed_trot, Cedtrot[x,y,:].real,  linestyle = "-.", label = "ED trotterized")
        cax.set_title(f"(i,j)=({x},{y})")


handles, labels = ax[0, 0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.94),
    ncol=2,
)

fig.tight_layout(rect=[0, 0, 1, 0.93])

plt.savefig("KitaevChain_spatial_corr_pfQM_trot.png", dpi = 200, bbox_inches = "tight")
plt.close()

In [211]:
taus = np.arange(0, dtau*(Ntau+1), dtau)

Ced  = np.load("ed_corr.npy")
taus_ed = np.load('ed_taus.npy')
Ced2 = np.load('ed_corr2.npy')

fig, ax = plt.subplots(L, L, figsize = (L*4.8, L*4.8))
fig.supylabel(r"$C_{ij}(\tau)$")
fig.supxlabel(r"$\tau$")
for x in range(L):
    for y in range(L):
        cax = ax[x, y]
        cax.grid()
        cax.errorbar(taus, Cd[x,y,:].real, marker = '.', capsize=3, label = "pfQMC", yerr = np.abs(Cd_err[x,y,:]), linestyle ="none")
        cax.errorbar(taus_ed, Ced[x,y,:].real,  linestyle = "--", label = "ED 0.2")
        cax.errorbar(taus_ed, Ced2[x,y,:].real,  linestyle = "--", label = "ED 0.4")
        cax.set_title(f"(i,j)=({x},{y})")


handles, labels = ax[0, 0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.94),
    ncol=2,
)

fig.tight_layout(rect=[0, 0, 1, 0.93])

plt.savefig("KitaevChain_spatial_corr_pfQM_ed_trot_double.png", dpi = 200, bbox_inches = "tight")
plt.close()